# Varying input to the model(s)
This example notebook provides an example of how to (systematically) manipulate or change the output of the preprocessing step, and feed the result input to the model. The example uses the vehicle sector, but the same principle can be applied to any of the other sectors.

First, we import everything that we need in the rest of the notebook:

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import prism
import warnings

from imagematerials.changedata import change_sector, ChangeAction, ChangeReplace
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMaterials, GenericStocks
from imagematerials.preprocessing import get_preprocessing_data


warnings.filterwarnings("ignore")

## Getting the preprocessing data, and changing it once
If we want to tweak the settings or the data that result from the preprocessing, before we feed it to the model, we can do so by using the `changedata` module. First we get the preprocessing data, like normal:

In [ ]:
# Get the preprocessing data for the vehicles sector
vhc_sector = get_preprocessing_data(
    "vehicles", Path("..", "data", "raw"), 
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)

In [ ]:
vhc_sector.prep_data

We can see that the output of the preprocessing is (essentially) a dictionary, with in its keys the data that will be used in the simulation. As long as we do not change the data type or dimensions of these inputs, we can change them in any way we want, and the simulation will work with the changed values.

To do this in a structured way, the `changedata` module exposes a `change_input()` function that takes the preprocessing result and a change definition to be applied to the preprocessing data. This change definition is a dictionary that mirrors the structure of the preprocessing data. The change defined under a given key in the change definition will be applied to the corresponding key in the preprocessing data.

> **Note:** The `change_input()` function operates on dictionaries. The output of `get_preprocessing_data()`, however, is a `Sector`, which wraps the underlying prep_data dictionary.
>
> For convenience, the `change_sector()` function can be used to manipulate a `Sector` object. This function uses `change_input()` in the background. Other than the type of the first argument, i.e. the data they are changing, both functions have the same signature. The `change_sector()` functions modifies `Sector` objects specifically, while the `change_input()` function can be used to modify any dictionary in the same way.

We now need to come up with a change definition. For this, we mirror the structure of the input dictionary, and place a `ChangeAction` in each position where we want to apply a change.

A `ChangeAction` is an object that exactly defines how the data stored in that key should be changed. Some have been predefined in the `changeinput` module, but you can also implement your own. In this example we will use the predefined `ChangeReplace` action, and define our own `ChangeFirstElementIn3DArray` action, that does not generalize very well.

In [ ]:
@dataclass
class ChangeFirstElementIn3DArray(ChangeAction):
    new_value: float

    def apply(self, value):
        value[0, 0, 0] = self.new_value
        return value

Now it is time to prepare our change definition. We will list all keys that exist in the preprocessing data, to show that this dictionary follows the same structure. However, only those keys that we will actually change, and have a ChangeAction for, will not be commented.

In [ ]:
change_definition = {
    "knowledge_graph": ChangeReplace(vhc_sector.prep_data['knowledge_graph']),  # replacing the knowledge_graph with a copy of itself (not that useful, but illustrates the process)
    # "lifetimes": None,  # - does not change
    # "maintenance_material_fractions": None,  # - does not change
    "material_fractions": ChangeFirstElementIn3DArray(0.42),  # change the first element to 0.42
    # "stocks": None,  # - does not change
    # "weights": None,  # - does not change
    # "set_unit_flexible": None  # - does not change
}

Passing this change definition to the `change_sector()` function, along with the original preprocessing data, will result in a modified `vhc_sector`.

In [ ]:
# Change the Sector data. By using inplace=True, the original dictionary will be modified.
# This is set to False by default, in which case a new dictionary / Sector will be returned from the function call.
change_sector(vhc_sector, change_definition, inplace=True)

To see the result of our work, let's look at the first element of the array that we manipulated using the `ChangeFirstElementToZero` action.

In [ ]:
vhc_sector.prep_data['material_fractions'][0, 0, 0]

## Creating and running the model

In [ ]:
# Define the timelines, including historic tail
complete_timeline = prism.Timeline(
    start=1960,
    end=2060,
    stepsize=1
)
simulation_timeline = prism.Timeline(
    start=1970,
    end=2060,
    stepsize=1
)

In [ ]:
# Create the model
factory = ModelFactory(
    vhc_sector, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    )
model = factory.finish()

In [ ]:
# Run the model
model.simulate(simulation_timeline)

## Visualize the output

In [ ]:
model.vehicles.get("stocks").sum("Region")

for type in model.vehicles.get("stocks").coords["Type"].values:
    model.vehicles.get("stocks").sel(Type=type).sum("Region").plot(label = type)

alu = model.vehicles.get("inflow_materials").to_array().sel(material = "aluminium").sum('Region')

for type in model.vehicles.get("inflow_materials").to_array().coords["Type"].values:
    alu.sel(Type=type).plot(label = type)

plt.legend()    

## Alternative use: systematically change the input to the model
We can use this setup to easily test out several different values for a given variable or datapoint. Using our not-so-useful but illustrative example `ChangeFirstElementIn3DArray`, we may want to test multiple values for the first element of the array. We can get the preprocessing data one time, and then use modified copies in several model runs.

In [ ]:
# Get the preprocessing data for the vehicles sector only once
vhc_sector = get_preprocessing_data(
    "vehicles", Path("..", "data", "raw"), 
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)

In [ ]:
list_of_values = [0.42, 0.41, 0.40, 0.39, 0.38]
for value in list_of_values:
    change_definition = {
        "material_fractions": ChangeFirstElementIn3DArray(value)
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition, inplace=False)
    print(
        f"Old value: {float(vhc_sector.prep_data['material_fractions'][0, 0, 0])};"
        f" new value: {float(new_vhc_sector.prep_data['material_fractions'][0, 0, 0])}."
    )
    # ... and then run the model.

As you can see, we make a new copy every time (the old value remains accessible, and unchanged), where the value has changed to one of the options from our list. We can use this new copy to run the model, capture the results, and compare the run to those using other values.